## Исследование влияния погодных условий на ДТП

Задача:

Выбрать 2 города в РФ, собрать по этим городам данные по погоде и ДТП с 2015-го по 2025-й год. Проанализировать эти данные и сделать выводы.

План:

1. Организовать процесс автоматического сбора(парсинг) всех городов РФ вместе с координатами
2. В supabase создать проект и таблицу с городами и координатами. Заргузить полученную таблицу с городами и координатами в supabase
4. Выбрать два города для дальнейшего исследования
5. Выгрузить погоду для двух городов с 2015 по 2025
6. Создать в supabase таблицу с погодой. Загрузить данные с погодой в supabase
7. Собрать данные по ДТП по выбранным городам с 2015 по 2025
8. Создать в supabase таблицу по ДТП. Загрузить данные по ДТП по выбранным городам в supabase
9. В supabase на базе таблице по ДТП создать основную таблицу по ДТП, Таблицу с деталями по ДТП, таблицу с данными по транспортным средствам, таблицу с участниками
10. в supabase создать пользователя для подключения к YandexDataLens
11. В Datalens создать новый проект, подключение к supabase. 
12. Загрузить данные с supabase, проанализировать, создать дашборды в Datalens
13. Написать общий вывод

### 1. Выгрузим список всех городов РФ с координатами

In [3]:
"""
СКРИПТ ДЛЯ ПАРСИНГА ВСЕХ ГОРОДОВ РОССИИ В CSV С КЕШИРОВАНИЕМ
Получает ВСЕ города России с Википедии (полный список) и добавляет координаты через Nominatim
Сохраняет ТОЛЬКО: City, Federal subject, Latitude, Longitude
УДАЛЯЕТ города без координат
"""

import os
import pandas as pd
from datetime import datetime, timedelta
from geopy.geocoders import Nominatim
import time
import requests
from io import StringIO
import re

# ========== КОНФИГУРАЦИЯ КЕША ==========
CACHE_FILE = 'russian_cities_all_with_coordinates.csv'
CACHE_MAX_AGE_DAYS = 30  # Обновлять кеш раз в месяц

def load_from_cache():
    """Проверяем и загружаем данные из кеша"""
    if os.path.exists(CACHE_FILE):
        # Проверяем возраст файла
        file_age = datetime.now() - datetime.fromtimestamp(os.path.getmtime(CACHE_FILE))
        
        if file_age < timedelta(days=CACHE_MAX_AGE_DAYS):
            print(f"✅ Загружаю данные из кеша (файлу {file_age.days} дней)")
            df = pd.read_csv(CACHE_FILE)
            print(f"📊 Загружено {len(df)} городов из кеша")
            return df
        else:
            print(f"🔄 Кеш устарел ({file_age.days} дней), обновляю...")
    else:
        print("🔄 Кеш не найден, начинаю парсинг...")
    return None

# ========== ОСНОВНАЯ ЛОГИКА ==========
print("Проверяю наличие актуального кеша...")

# Пытаемся загрузить из кеша
cached_df = load_from_cache()

if cached_df is not None:
    # Если кеш свежий — используем его и завершаем скрипт
    print("\n✅ Использую данные из кеша!")
    print("📋 Содержимое кеша (первые 10 строк):")
    print(cached_df.head(10))
    
    # Показываем статистику кеша
    print(f"\n📊 СТАТИСТИКА КЕША:")
    print(f"   • Всего городов: {len(cached_df)}")
    print(f"   • Уникальных субъектов: {cached_df['Federal subject'].nunique()}")
    print(f"   • Широта: {cached_df['Latitude'].min():.2f}° - {cached_df['Latitude'].max():.2f}°")
    print(f"   • Долгота: {cached_df['Longitude'].min():.2f}° - {cached_df['Longitude'].max():.2f}°")
    
    print(f"\n📁 Файл кеша: {CACHE_FILE}")
    print("✨ Скрипт завершен, данные взяты из кеша!")
    print("💡 Чтобы обновить кеш, удалите файл или подождите 30 дней.")
    
    # В Jupyter Notebook просто завершаем выполнение
    # Если запускаете как скрипт, используйте: exit()
    
else:
    # ========== ЕСЛИ КЕША НЕТ ИЛИ ОН УСТАРЕЛ ==========
    # ТОЛЬКО ТОГДА выполняем долгий парсинг
    
    print("\n" + "="*60)
    print("🔄 Кеш не найден или устарел, начинаю парсинг ВСЕХ городов...")
    print("="*60)

    print(" Начинаю парсинг ВСЕХ городов России...")

    # 1. ЗАГРУЖАЕМ ПОЛНЫЙ СПИСОК ВСЕХ ГОРОДОВ РОССИИ
    print("\n📥 Загружаю ПОЛНЫЙ список всех городов России с Википедии...")

    try:
        # ИСПОЛЬЗУЕМ СТРАНИЦУ С ПОЛНЫМ СПИСКОМ ВСЕХ ГОРОДОВ
        url = "https://en.wikipedia.org/wiki/List_of_cities_and_towns_in_Russia"
        
        print(f"🌐 Источник данных: {url}")
        
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }
        
        response = requests.get(url, headers=headers, timeout=20)
        response.raise_for_status()
        
        html_content = StringIO(response.text)
        
        # Пробуем загрузить все таблицы
        tables = pd.read_html(html_content)
        
        print(f"✅ Найдено {len(tables)} таблиц на странице")
        
        if tables:
            # На странице обычно несколько таблиц по регионам
            # Нужно объединить их все
            all_cities_data = []
            
            for table_idx, table in enumerate(tables):
                print(f"\n📊 Анализирую таблицу {table_idx + 1} ({len(table)} строк)...")
                
                # Показываем структуру таблицы
                print(f"   Колонки: {list(table.columns)}")
                if len(table) > 0:
                    print(f"   Пример данных: {table.iloc[0].tolist()[:3]}...")
                
                # Проверяем, что таблица содержит данные о городах
                # Ищем столбец с названиями городов
                city_col = None
                for col in table.columns:
                    col_str = str(col).lower()
                    # Проверяем различные варианты названий столбцов
                    if any(keyword in col_str for keyword in ['city', 'town', 'название', 'город', 'name', 'settlement']):
                        city_col = col
                        break
                
                # Если не нашли по ключевым словам, пробуем определить по содержимому
                if city_col is None and len(table.columns) > 0:
                    # Проверяем первую колонку - часто это названия городов
                    first_col_samples = table.iloc[:5, 0].astype(str).tolist()
                    if any(sample and len(sample.strip()) > 1 for sample in first_col_samples):
                        city_col = table.columns[0]
                        print(f"   🤔 Использую первую колонку '{city_col}' как City (по умолчанию)")
                
                if city_col is not None:
                    # Создаем временный DataFrame для этой таблицы
                    temp_data = []
                    
                    for idx, row in table.iterrows():
                        try:
                            city_name = str(row[city_col]).strip()
                            
                            # Очищаем название города от ссылок и примечаний
                            city_name = re.sub(r'\[.*?\]|\(.*?\)', '', city_name)
                            city_name = city_name.split('!')[0].strip()
                            
                            if city_name and city_name.lower() not in ['nan', 'none', '']:
                                # Ищем субъект федерации
                                subject = "Не указан"
                                
                                # Пробуем найти столбец с субъектом
                                for col in table.columns:
                                    col_str = str(col).lower()
                                    if any(keyword in col_str for keyword in ['subject', 'federal', 'region', 'субъект', 'область', 'республика']):
                                        subject_val = str(row[col]).strip()
                                        if subject_val and subject_val.lower() not in ['nan', 'none', '']:
                                            subject = re.sub(r'\[.*?\]|\(.*?\)', '', subject_val)
                                            break
                                
                                temp_data.append({
                                    'City': city_name,
                                    'Federal subject': subject
                                })
                        except Exception as e:
                            continue  # Пропускаем проблемные строки
                    
                    if temp_data:
                        temp_df = pd.DataFrame(temp_data)
                        all_cities_data.append(temp_df)
                        print(f"   ✅ Добавлено {len(temp_data)} городов из таблицы {table_idx + 1}")
                    else:
                        print(f"   ⚠️  Не удалось извлечь данные из таблицы {table_idx + 1}")
            
            # Объединяем все данные
            if all_cities_data:
                df = pd.concat(all_cities_data, ignore_index=True)
                
                # Удаляем дубликаты
                initial_count = len(df)
                df = df.drop_duplicates(subset=['City'], keep='first')
                print(f"\n🧹 Удалено {initial_count - len(df)} дубликатов городов")
                
                print(f"✅ Итого загружено {len(df)} уникальных городов")
                
                # Показываем первые 15 городов для проверки
                print("\n📋 Первые 15 городов из полного списка:")
                print(df.head(15))
                
                # Показываем статистику по субъектам
                subject_stats = df['Federal subject'].value_counts()
                print(f"\n📊 Уникальных субъектов федерации: {len(subject_stats)}")
                print("📍 Примеры субъектов:")
                for subject, count in subject_stats.head(10).items():
                    print(f"   • {subject}: {count} городов")
                
            else:
                raise Exception("Не удалось извлечь данные о городах")
        else:
            raise Exception("Таблицы не найдены на странице")
            
    except Exception as e:
        print(f"❌ Ошибка при загрузке полного списка: {e}")
        print("\n📝 Создаю тестовый список городов...")
        
        # Тестовые данные - список реальных городов России
        test_data = {
            'City': [
                'Moscow', 'Saint Petersburg', 'Novosibirsk', 'Yekaterinburg', 'Kazan',
                'Nizhny Novgorod', 'Chelyabinsk', 'Samara', 'Omsk', 'Rostov-on-Don',
                'Ufa', 'Krasnoyarsk', 'Perm', 'Voronezh', 'Volgograd',
                'Saratov', 'Krasnodar', 'Tolyatti', 'Izhevsk', 'Barnaul',
                'Vladivostok', 'Irkutsk', 'Khabarovsk', 'Yaroslavl', 'Makhachkala',
                'Tomsk', 'Orenburg', 'Kemerovo', 'Novokuznetsk', 'Ryazan',
                'Astrakhan', 'Penza', 'Lipetsk', 'Kirov', 'Cheboksary',
                'Tula', 'Kaliningrad', 'Bryansk', 'Ivanovo', 'Magnitogorsk',
                'Kursk', 'Tver', 'Nizhny Tagil', 'Stavropol', 'Ulyanovsk',
                'Arkhangelsk', 'Belgorod', 'Kurgan', 'Smolensk', 'Kaluga',
                'Orël', 'Volzhsky', 'Murmansk', 'Vladimir', 'Surgut',
                'Chita', 'Vologda', 'Yakutsk', 'Sochi', 'Grozny'
            ],
            'Federal subject': [
                'Moscow', 'Saint Petersburg', 'Novosibirsk Oblast', 'Sverdlovsk Oblast', 'Republic of Tatarstan',
                'Nizhny Novgorod Oblast', 'Chelyabinsk Oblast', 'Samara Oblast', 'Omsk Oblast', 'Rostov Oblast',
                'Republic of Bashkortostan', 'Krasnoyarsk Krai', 'Perm Krai', 'Voronezh Oblast', 'Volgograd Oblast',
                'Saratov Oblast', 'Krasnodar Krai', 'Samara Oblast', 'Udmurt Republic', 'Altai Krai',
                'Primorsky Krai', 'Irkutsk Oblast', 'Khabarovsk Krai', 'Yaroslavl Oblast', 'Republic of Dagestan',
                'Tomsk Oblast', 'Orenburg Oblast', 'Kemerovo Oblast', 'Kemerovo Oblast', 'Ryazan Oblast',
                'Astrakhan Oblast', 'Penza Oblast', 'Lipetsk Oblast', 'Kirov Oblast', 'Chuvash Republic',
                'Tula Oblast', 'Kaliningrad Oblast', 'Bryansk Oblast', 'Ivanovo Oblast', 'Chelyabinsk Oblast',
                'Kursk Oblast', 'Tver Oblast', 'Sverdlovsk Oblast', 'Stavropol Krai', 'Ulyanovsk Oblast',
                'Arkhangelsk Oblast', 'Belgorod Oblast', 'Kurgan Oblast', 'Smolensk Oblast', 'Kaluga Oblast',
                'Oryol Oblast', 'Volgograd Oblast', 'Murmansk Oblast', 'Vladimir Oblast', 'Khanty-Mansi Autonomous Okrug',
                'Zabaykalsky Krai', 'Vologda Oblast', 'Sakha Republic', 'Krasnodar Krai', 'Chechen Republic'
            ]
        }
        df = pd.DataFrame(test_data)
        print(f"✅ Создан тестовый список из {len(df)} городов")

    # 2. ПОЛУЧАЕМ КООРДИНАТЫ ЧЕРЕЗ NOMINATIM
    print("\n" + "="*60)
    print("🗺️  Начинаю получение координат через Nominatim...")
    print("="*60)

    geolocator = Nominatim(
        user_agent="russian_cities_full_parser_v1.0",
        timeout=25
    )

    # Списки для координат
    latitudes = []
    longitudes = []
    failed_cities = []

    total_cities = len(df)
    print(f"\n⏳ Обрабатываю {total_cities} городов...")
    print(f"   ⚠️  Это займет примерно {total_cities} секунд ({total_cities//60} минут)")
    print("   Каждый запрос - 1 секунда (ограничение Nominatim)")
    print("\n📊 Статистика загрузки:")
    print("   • Уникальных городов: " + str(total_cities))
    print("   • Примерное время: " + f"{total_cities//60} минут {total_cities%60} секунд")

    # Счетчики
    found_coords = 0
    not_found = 0

    for i, row in df.iterrows():
        city_name = row['City']
        federal_subject = row['Federal subject']
        
        # Показываем прогресс
        progress = f"{i+1}/{total_cities}"
        
        # Показываем информацию о городе
        if (i < 10) or (i >= total_cities - 10) or ((i + 1) % 100 == 0):
            print(f"\n{progress}: 🏙️  {city_name}")
            if federal_subject != "Не указан":
                print(f"      📍 {federal_subject}")
        
        try:
            # Формируем запрос для поиска
            search_queries = []
            
            # Сначала пробуем с указанием субъекта
            if federal_subject != "Не указан":
                search_queries.append(f"{city_name}, {federal_subject}, Russia")
                search_queries.append(f"{city_name}, {federal_subject}")
            
            # Затем пробуем общие варианты
            search_queries.append(f"{city_name}, Russia")
            search_queries.append(f"{city_name}")
            
            location = None
            for query in search_queries:
                try:
                    location = geolocator.geocode(query, timeout=15, language='en')
                    if location:
                        break
                except:
                    continue
            
            if location:
                latitudes.append(location.latitude)
                longitudes.append(location.longitude)
                found_coords += 1
                
                if (i < 10) or (i >= total_cities - 10) or ((i + 1) % 100 == 0):
                    print(f"      ✅ Координаты: {location.latitude:.6f}, {location.longitude:.6f}")
                    print(f"      📍 Найден по запросу: {query}")
            else:
                latitudes.append(None)
                longitudes.append(None)
                not_found += 1
                failed_cities.append(city_name)
                
                if (i < 10) or (i >= total_cities - 10) or ((i + 1) % 100 == 0):
                    print(f"      ❌ Координаты не найдены")
        
        except Exception as e:
            latitudes.append(None)
            longitudes.append(None)
            not_found += 1
            failed_cities.append(city_name)
            
            if (i < 10) or (i >= total_cities - 10) or ((i + 1) % 100 == 0):
                print(f"      ⚠️  Ошибка: {str(e)[:50]}...")
        
        # Пауза между запросами (соблюдаем правила Nominatim)
        time.sleep(1.0)
        
        # Показываем промежуточную статистику каждые 50 городов
        if (i + 1) % 50 == 0 and (i + 1) < total_cities:
            print(f"\n📊 Промежуточный результат ({i+1}/{total_cities}):")
            print(f"   ✅ Найдено: {found_coords}")
            print(f"   ❌ Не найдено: {not_found}")
            success_rate = (found_coords / (i + 1)) * 100
            print(f"   📈 Успешность: {success_rate:.1f}%")

    # 3. ДОБАВЛЯЕМ КООРДИНАТЫ В ТАБЛИЦУ
    df['Latitude'] = latitudes
    df['Longitude'] = longitudes

    # 4. УДАЛЯЕМ СТРОКИ БЕЗ КООРДИНАТ
    print("\n🔄 Фильтрую данные...")

    # Сохраняем города без координат в отдельный файл
    failed_df = df[df['Latitude'].isna() | df['Longitude'].isna()].copy()
    failed_count = len(failed_df)

    # Удаляем строки, где Latitude или Longitude равны None
    initial_count = len(df)
    clean_df = df.dropna(subset=['Latitude', 'Longitude']).copy()
    final_count = len(clean_df)
    removed_count = initial_count - final_count

    print(f"✅ Всего обработано городов: {initial_count}")
    print(f"✅ Сохранено с координатами: {final_count} ({final_count/initial_count*100:.1f}%)")
    print(f"❌ Удалено без координат: {removed_count} ({removed_count/initial_count*100:.1f}%)")

    if failed_count > 0:
        print(f"\n⚠️  Города без координат сохранены в отдельный файл: russian_cities_no_coords.csv")
        failed_df[['City', 'Federal subject']].to_csv('russian_cities_no_coords.csv', index=False, encoding='utf-8-sig')

    # 5. СОХРАНЯЕМ ОСНОВНОЙ CSV ФАЙЛ (КЕШ!)
    csv_filename = CACHE_FILE  # Используем имя из конфигурации кеша
    clean_df.to_csv(csv_filename, index=False, encoding='utf-8-sig')

    print(f"\n💾 Основной файл сохранен как КЕШ: {csv_filename}")
    print(f"💡 При следующем запуске (в течение 30 дней) данные будут загружены отсюда!")

    # 6. ПОКАЗЫВАЕМ РЕЗУЛЬТАТ
    print("\n" + "="*60)
    print("📊 ИТОГОВЫЕ РЕЗУЛЬТАТЫ ПАРСИНГА")
    print("="*60)

    print(f"\n📈 СТАТИСТИКА:")
    print(f"   Всего городов в источнике: {initial_count}")
    print(f"   Успешно получены координаты: {final_count}")
    print(f"   Не найдены координаты: {removed_count}")
    print(f"   Процент успеха: {(final_count/initial_count)*100:.1f}%")

    print("\n📋 ПЕРВЫЕ 20 СТРОК CSV ФАЙЛА:")
    print(clean_df.head(20))

    print("\n📋 ПОСЛЕДНИЕ 20 СТРОК CSV ФАЙЛА:")
    print(clean_df.tail(20))

    # Группировка по субъектам федерации
    subject_counts = clean_df['Federal subject'].value_counts()
    print(f"\n📊 РАСПРЕДЕЛЕНИЕ ПО СУБЪЕКТАМ ФЕДЕРАЦИИ:")
    print(f"   Всего субъектов: {len(subject_counts)}")

    print("\n📍 ТОП-15 субъектов по количеству городов:")
    for subject, count in subject_counts.head(15).items():
        percentage = (count / final_count) * 100
        print(f"   • {subject}: {count} городов ({percentage:.1f}%)")

    print("\n📍 Субъекты с 1 городом:")
    single_city_subjects = subject_counts[subject_counts == 1].index.tolist()
    if single_city_subjects:
        print(f"   Всего {len(single_city_subjects)} субъектов")
        if len(single_city_subjects) <= 10:
            for subject in single_city_subjects[:10]:
                print(f"   • {subject}")

    print("\n📊 ГЕОГРАФИЧЕСКАЯ СТАТИСТИКА:")
    if final_count > 0:
        print(f"   🗺️  Широта: от {clean_df['Latitude'].min():.4f}° до {clean_df['Latitude'].max():.4f}°")
        print(f"   🗺️  Долгота: от {clean_df['Longitude'].min():.4f}° до {clean_df['Longitude'].max():.4f}°")
        
        # Находим крайние города
        min_lat_city = clean_df.loc[clean_df['Latitude'].idxmin(), 'City']
        max_lat_city = clean_df.loc[clean_df['Latitude'].idxmax(), 'City']
        min_lon_city = clean_df.loc[clean_df['Longitude'].idxmin(), 'City']
        max_lon_city = clean_df.loc[clean_df['Longitude'].idxmax(), 'City']
        
        print(f"\n📍 ГЕОГРАФИЧЕСКИЕ ЭКСТРЕМУМЫ:")
        print(f"   • Самый южный: {min_lat_city} ({clean_df['Latitude'].min():.4f}°)")
        print(f"   • Самый северный: {max_lat_city} ({clean_df['Latitude'].max():.4f}°)")
        print(f"   • Самый западный: {min_lon_city} ({clean_df['Longitude'].min():.4f}°)")
        print(f"   • Самый восточный: {max_lon_city} ({clean_df['Longitude'].max():.4f}°)")

    print("\n" + "="*60)
    print("✅ ПАРСИНГ ВСЕХ ГОРОДОВ РОССИИ ЗАВЕРШЕН!")
    print("="*60)

    print(f"\n📁 СОЗДАННЫЕ ФАЙЛЫ:")
    print(f"   1. {csv_filename} - {final_count} городов с координатами (ОСНОВНОЙ КЕШ)")
    if failed_count > 0:
        print(f"   2. russian_cities_no_coords.csv - {failed_count} городов без координат")

    print(f"\n📊 ИНФОРМАЦИЯ О ДАННЫХ:")
    print(f"   • Всего строк: {final_count}")
    print(f"   • Колонки: City, Federal subject, Latitude, Longitude")
    print(f"   • Кодировка: UTF-8 with BOM (для корректного отображения в Excel)")
    print(f"   • Разделитель: запятая")
    print(f"   • Формат координат: десятичные градусы")

    print(f"\n✨ Готово! Теперь у вас есть полный список городов России с координатами.")
    print(f"💾 Данные сохранены в кеш. При следующем запуске они будут загружены мгновенно!")
    print(f"🔍 Файл готов для использования в GIS-системах, аналитике и визуализации.")

Проверяю наличие актуального кеша...
✅ Загружаю данные из кеша (файлу 21 дней)
📊 Загружено 0 городов из кеша

✅ Использую данные из кеша!
📋 Содержимое кеша (первые 10 строк):
Empty DataFrame
Columns: [City, Federal subject, Latitude, Longitude]
Index: []

📊 СТАТИСТИКА КЕША:
   • Всего городов: 0
   • Уникальных субъектов: 0
   • Широта: nan° - nan°
   • Долгота: nan° - nan°

📁 Файл кеша: russian_cities_all_with_coordinates.csv
✨ Скрипт завершен, данные взяты из кеша!
💡 Чтобы обновить кеш, удалите файл или подождите 30 дней.


### 2. Загрузим список городов с координатами в supabase

In [8]:
import pandas as pd
import requests
import json

# Ваши данные
SUPABASE_URL = "https://xrtiwzpkjapblwcmkyve.supabase.co"
SUPABASE_KEY = "sb_publishable_9qNc84zjkn-20CUJhN_8EQ_oxdOSCzC"

# Загружаем CSV
df = pd.read_csv('russian_cities_all_with_coordinates.csv')

# Переименовываем колонки для Supabase
df = df.rename(columns={
    'City': 'city',
    'Federal subject': 'federal_subject', 
    'Latitude': 'latitude',
    'Longitude': 'longitude'
})

# Конвертируем в JSON
data = df.to_dict('records')

print(f"📊 Готово загрузить {len(data)} городов")

# Отправляем данные
headers = {
    'apikey': SUPABASE_KEY,
    'Authorization': f'Bearer {SUPABASE_KEY}',
    'Content-Type': 'application/json',
    'Prefer': 'resolution=merge-duplicates'  # Для upsert
}

# Отправляем пакетами по 100
batch_size = 100
for i in range(0, len(data), batch_size):
    batch = data[i:i+batch_size]
    
    response = requests.post(
        f"{SUPABASE_URL}/rest/v1/russian_cities",
        headers=headers,
        json=batch
    )
    
    if response.status_code in [200, 201]:
        print(f"✅ Загружено {len(batch)} записей")
    elif response.status_code == 409:
        print(f"⚠️  Таблица уже существует, обновляю...")
        # Пробуем через PATCH для обновления
        response = requests.patch(
            f"{SUPABASE_URL}/rest/v1/russian_cities",
            headers=headers,
            json=batch
        )
        if response.status_code == 200:
            print(f"✅ Обновлено {len(batch)} записей")
        else:
            print(f"❌ Ошибка: {response.text[:100]}")
    else:
        print(f"❌ Ошибка {response.status_code}: {response.text[:100]}")

print("\n✨ Загрузка завершена!")

📊 Готово загрузить 1101 городов
✅ Загружено 100 записей
✅ Загружено 100 записей
✅ Загружено 100 записей
✅ Загружено 100 записей
✅ Загружено 100 записей
✅ Загружено 100 записей
✅ Загружено 100 записей
✅ Загружено 100 записей
✅ Загружено 100 записей
✅ Загружено 100 записей
✅ Загружено 100 записей
✅ Загружено 1 записей

✨ Загрузка завершена!


### 3. Выбирем два города для дальнейшего исследования

In [12]:
import pandas as pd

# Загружаем CSV файл
df = pd.read_csv('russian_cities_all_with_coordinates.csv')

# Ищем города
kazan = df[df['City'].str.contains('Kazan|Казань', case=False, na=False)].head(1)
penza = df[df['City'].str.contains('Penza|Пенза', case=False, na=False)].head(1)

# Объединяем найденные города
selected_cities = pd.concat([kazan, penza], ignore_index=True)

# Сохраняем результат
selected_cities.to_csv('selected_cities.csv', index=False, encoding='utf-8-sig')
print(f"✅ Файл сохранён. Найдено городов: {len(selected_cities)}")

# Краткая информация
print("\nВыбранные города:")
for _, row in selected_cities.iterrows():
    print(f"{row['City']} - {row['Federal subject']} ({row['Latitude']}, {row['Longitude']})")

✅ Файл сохранён. Найдено городов: 2

Выбранные города:
Kazan - Republic of Tatarstan (55.7946485, 49.1115022)
Penza - Penza Oblast (53.1953477, 45.0190437)


### 4. Выгрузим погоду по двум городам с 2015 по 2025

Погода с 2015 по 2025

In [7]:
import pandas as pd
from datetime import datetime, timedelta
import requests
import time
import os
import json
import hashlib
import random

print("🌤️ Начинаю сбор исторических почасовых данных погоды...")

# 1. ЗАГРУЖАЕМ ГОРОДА ИЗ CSV ФАЙЛА
print("\n📥 Загружаю города из selected_cities.csv...")

try:
    if not os.path.exists('selected_cities.csv'):
        raise FileNotFoundError("Файл selected_cities.csv не найден")
    
    cities_df = pd.read_csv('selected_cities.csv')
    
    print(f"📊 Загружено {len(cities_df)} городов")
    print(f"📋 Колонки: {list(cities_df.columns)}")
    
    cities = []
    
    city_col = None
    lat_col = None
    lon_col = None
    subject_col = None
    
    for col in cities_df.columns:
        col_lower = str(col).lower()
        
        if 'city' in col_lower or 'город' in col_lower or 'name' in col_lower:
            city_col = col
        elif 'lat' in col_lower or 'широта' in col_lower:
            lat_col = col
        elif 'lon' in col_lower or 'long' in col_lower or 'долгота' in col_lower:
            lon_col = col
        elif 'federal' in col_lower or 'subject' in col_lower or 'субъект' in col_lower:
            subject_col = col
    
    if city_col is None and len(cities_df.columns) > 0:
        city_col = cities_df.columns[0]
    
    if lat_col is None and len(cities_df.columns) > 1:
        lat_col = cities_df.columns[1]
    
    if lon_col is None and len(cities_df.columns) > 2:
        lon_col = cities_df.columns[2]
    
    for idx, row in cities_df.iterrows():
        try:
            city_name = str(row[city_col]) if city_col else f"City_{idx}"
            latitude = float(row[lat_col]) if lat_col and pd.notna(row[lat_col]) else None
            longitude = float(row[lon_col]) if lon_col and pd.notna(row[lon_col]) else None
            
            if latitude is not None and longitude is not None:
                city_info = {
                    'name': city_name,
                    'lat': latitude,
                    'lon': longitude,
                    'index': idx
                }
                
                if subject_col and subject_col in row:
                    city_info['federal_subject'] = str(row[subject_col])
                else:
                    city_info['federal_subject'] = "Не указан"
                
                cities.append(city_info)
                
        except Exception as e:
            print(f"⚠️  Ошибка обработки строки {idx}: {e}")
    
    print(f"\n📊 Итого для обработки: {len(cities)} городов с координатами")
    
    if len(cities) == 0:
        print("❌ Нет городов с координатами для обработки")
        exit()
        
except Exception as e:
    print(f"❌ Ошибка загрузки CSV: {e}")
    print("\n📝 Создаю тестовые данные...")
    
    cities = [
        {'name': 'Moscow', 'lat': 55.7558, 'lon': 37.6176, 'federal_subject': 'Moscow'},
        {'name': 'Penza', 'lat': 53.2007, 'lon': 45.0046, 'federal_subject': 'Penza Oblast'}
    ]

print("\n" + "="*60)
print("🌍 СПИСОК ГОРОДОВ ДЛЯ СБОРА ДАННЫХ:")
print("="*60)
for city in cities:
    print(f"📍 {city['name']}: {city['lat']:.6f}, {city['lon']:.6f} ({city.get('federal_subject', 'Не указан')})")

# 2. СОЗДАЕМ ПАПКУ ДЛЯ КЭША
CACHE_DIR = "weather_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

def get_cache_key(city_name, lat, lon, start_date, end_date):
    key_string = f"{city_name}_{lat}_{lon}_{start_date}_{end_date}"
    return hashlib.md5(key_string.encode()).hexdigest()

def get_cached_weather(cache_key):
    cache_file = os.path.join(CACHE_DIR, f"{cache_key}.pkl")
    if os.path.exists(cache_file):
        try:
            df = pd.read_pickle(cache_file)
            print(f"   📂 Загружено из кэша ({len(df)} записей)")
            return df
        except Exception as e:
            print(f"   ⚠️  Ошибка чтения кэша: {e}")
    return None

def save_to_cache(df, cache_key):
    cache_file = os.path.join(CACHE_DIR, f"{cache_key}.pkl")
    try:
        df.to_pickle(cache_file)
        print(f"   💾 Сохранено в кэш")
    except Exception as e:
        print(f"   ⚠️  Ошибка сохранения кэша: {e}")

# 3. УЛУЧШЕННАЯ ФУНКЦИЯ С ПОВТОРНЫМИ ПОПЫТКАМИ
def get_hourly_weather_with_retry(city_name, lat, lon, start_date, end_date, federal_subject, max_retries=5):
    """Получение данных с повторными попытками при ошибках"""
    
    # Проверяем, не является ли конечная дата будущей
    today = datetime.now().date()
    end_date_obj = datetime.strptime(end_date, '%Y-%m-%d').date()
    
    if end_date_obj > today:
        print(f"   ⚠️  Конечная дата {end_date} в будущем. Использую {today} как конечную дату")
        end_date = today.strftime('%Y-%m-%d')
        
        if start_date > end_date:
            print(f"   ⚠️  Начальная дата {start_date} позже конечной {end_date}. Пропускаем")
            return None
    
    cache_key = get_cache_key(city_name, lat, lon, start_date, end_date)
    
    # Проверяем кэш
    cached_data = get_cached_weather(cache_key)
    if cached_data is not None:
        if 'city' not in cached_data.columns:
            cached_data.insert(0, 'city', city_name)
            cached_data.insert(1, 'federal_subject', federal_subject)
            cached_data.insert(2, 'latitude', lat)
            cached_data.insert(3, 'longitude', lon)
            cached_data['date'] = pd.to_datetime(cached_data['time']).dt.date
        return cached_data
    
    hourly_params = [
        'temperature_2m', 'relative_humidity_2m', 'dew_point_2m',
        'apparent_temperature', 'pressure_msl', 'surface_pressure',
        'precipitation', 'rain', 'snowfall', 'cloud_cover',
        'cloud_cover_low', 'cloud_cover_mid', 'cloud_cover_high',
        'shortwave_radiation', 'direct_radiation', 'diffuse_radiation',
        'direct_normal_irradiance', 'global_tilted_irradiance',
        'terrestrial_radiation', 'wind_speed_10m', 'wind_speed_100m',
        'wind_direction_10m', 'wind_direction_100m', 'wind_gusts_10m',
        'et0_fao_evapotranspiration', 'soil_temperature_0_to_7cm',
        'soil_temperature_7_to_28cm', 'soil_temperature_28_to_100cm',
        'soil_temperature_100_to_255cm', 'soil_moisture_0_to_7cm',
        'soil_moisture_7_to_28cm', 'soil_moisture_28_to_100cm',
        'soil_moisture_100_to_255cm'
    ]
    
    params = {
        'latitude': lat,
        'longitude': lon,
        'start_date': start_date,
        'end_date': end_date,
        'hourly': hourly_params,
        'timezone': 'Europe/Moscow',
        'temperature_unit': 'celsius',
        'wind_speed_unit': 'kmh',
        'precipitation_unit': 'mm'
    }
    
    for attempt in range(max_retries):
        try:
            print(f"\n🌤️ Запрашиваю данные для {city_name}... (попытка {attempt + 1}/{max_retries})")
            print(f"   📅 Период: {start_date} - {end_date}")
            
            # Добавляем случайную задержку между попытками
            if attempt > 0:
                wait_time = (2 ** attempt) + random.uniform(1, 3)
                print(f"   ⏳ Ожидание {wait_time:.1f} секунд перед повторной попыткой...")
                time.sleep(wait_time)
            
            response = requests.get("https://archive-api.open-meteo.com/v1/archive", params=params, timeout=60)
            
            if response.status_code == 429:
                print(f"⚠️  Слишком много запросов. Ожидание перед повторной попыткой...")
                time.sleep(10 * (attempt + 1))  # Увеличиваем задержку
                continue
                
            response.raise_for_status()
            data = response.json()
            
            if 'hourly' not in data or not data['hourly']:
                print(f"⚠️  Для {city_name} нет данных за период {start_date} - {end_date}")
                return None
            
            df = pd.DataFrame(data['hourly'])
            
            if len(df) == 0:
                return None
            
            df.insert(0, 'city', city_name)
            df.insert(1, 'federal_subject', federal_subject)
            df.insert(2, 'latitude', lat)
            df.insert(3, 'longitude', lon)
            df['date'] = pd.to_datetime(df['time']).dt.date
            
            print(f"✅ {city_name}: получено {len(df)} записей")
            
            # Сохраняем в кэш
            save_to_cache(df, cache_key)
            
            return df
            
        except requests.exceptions.RequestException as e:
            if hasattr(response, 'status_code') and response.status_code == 400:
                print(f"❌ Ошибка 400: Неверный запрос. Возможно, период включает будущие даты")
                return None
            print(f"❌ Ошибка сети: {e}")
            if attempt == max_retries - 1:
                return None
            time.sleep(5 * (attempt + 1))
            
        except Exception as e:
            print(f"❌ Ошибка: {e}")
            if attempt == max_retries - 1:
                return None
            time.sleep(5)
    
    return None

# 4. ФУНКЦИЯ ДЛЯ РАЗБИВКИ ПЕРИОДА НА ЧАСТИ (ИСПРАВЛЕННАЯ)
def get_weather_full_period_chunked(city_name, lat, lon, federal_subject, chunk_years=1):
    """Получение данных за полный период с разбивкой на небольшие части"""
    start_date = datetime(2015, 1, 1)  # ИЗМЕНЕНО: с 2015 года
    # Ограничиваем конечную дату 31 декабря 2025 года
    end_date = datetime(2025, 12, 31)  # ИЗМЕНЕНО: до 2025 года
    
    # Проверяем, не превышает ли конечная дата сегодняшний день
    today = datetime.now()
    if end_date > today:
        print(f"\n⚠️  Внимание: запрошен период до {end_date.strftime('%Y-%m-%d')}, но сегодня {today.strftime('%Y-%m-%d')}")
        print(f"   Данные за будущие периоды ({today.strftime('%Y-%m-%d')} - {end_date.strftime('%Y-%m-%d')}) недоступны")
        end_date = today
        print(f"   Использую период до {end_date.strftime('%Y-%m-%d')}")
    
    print(f"\n📅 Полный период: 2015-01-01 - {end_date.strftime('%Y-%m-%d')}")
    print(f"   Разбиваю на части по {chunk_years} году(а)")
    
    all_dfs = []
    current_date = start_date
    
    total_chunks = 0
    successful_chunks = 0
    failed_chunks = []
    
    while current_date <= end_date:
        # Вычисляем конец текущего чанка
        chunk_end = current_date + timedelta(days=int(365 * chunk_years) - 1)
        if chunk_end > end_date:
            chunk_end = end_date
        
        start_str = current_date.strftime('%Y-%m-%d')
        end_str = chunk_end.strftime('%Y-%m-%d')
        
        print(f"\n   📦 Часть {total_chunks + 1}: {start_str} - {end_str}")
        
        # Добавляем задержку между запросами
        if total_chunks > 0:
            delay = random.uniform(2, 4)
            print(f"   ⏳ Задержка {delay:.1f} сек...")
            time.sleep(delay)
        
        df_chunk = get_hourly_weather_with_retry(
            city_name, lat, lon, start_str, end_str, federal_subject
        )
        
        if df_chunk is not None:
            all_dfs.append(df_chunk)
            successful_chunks += 1
            print(f"   ✅ Часть {total_chunks + 1} успешно загружена")
        else:
            failed_chunks.append((start_str, end_str))
            print(f"   ❌ Часть {total_chunks + 1} не удалось загрузить")
        
        total_chunks += 1
        
        # Переходим к следующему чанку
        current_date = chunk_end + timedelta(days=1)
    
    # Повторная попытка для неудачных частей
    if failed_chunks:
        print(f"\n🔄 Повторная попытка для {len(failed_chunks)} неудачных частей...")
        time.sleep(10)  # Длительная пауза перед повторными попытками
        
        for start_str, end_str in failed_chunks:
            print(f"\n   🔄 Повторная попытка: {start_str} - {end_str}")
            time.sleep(random.uniform(3, 5))
            
            df_chunk = get_hourly_weather_with_retry(
                city_name, lat, lon, start_str, end_str, federal_subject, max_retries=3
            )
            
            if df_chunk is not None:
                all_dfs.append(df_chunk)
                successful_chunks += 1
                print(f"   ✅ Повторная попытка успешна")
    
    if all_dfs:
        combined_df = pd.concat(all_dfs, ignore_index=True)
        combined_df = combined_df.drop_duplicates(subset=['time'], keep='first')
        combined_df = combined_df.sort_values('time')
        
        print(f"\n✅ {city_name}: всего собрано {len(combined_df)} записей из {successful_chunks}/{total_chunks} частей")
        
        # Сохраняем объединенные данные в кэш
        cache_key = get_cache_key(city_name, lat, lon, '2015-01-01', end_date.strftime('%Y-%m-%d'))
        save_to_cache(combined_df, cache_key)
        
        return combined_df
    
    return None

# 5. ФУНКЦИЯ ДЛЯ ПОЛУЧЕНИЯ ВСЕХ ГОРОДОВ ИЗ КЭША
def load_all_cached_data():
    all_cached_data = []
    cache_files = [f for f in os.listdir(CACHE_DIR) if f.endswith('.pkl')]
    
    if not cache_files:
        print("📭 Кэш пуст")
        return None
    
    print(f"\n📂 Найдено {len(cache_files)} файлов в кэше")
    
    for i, cache_file in enumerate(cache_files, 1):
        try:
            df = pd.read_pickle(os.path.join(CACHE_DIR, cache_file))
            all_cached_data.append(df)
            city_name = df['city'].iloc[0] if 'city' in df.columns else f"Cache_{i}"
            print(f"   {i:3d}. {cache_file[:20]}... - {city_name} ({len(df)} записей)")
        except Exception as e:
            print(f"   ⚠️  Ошибка загрузки {cache_file}: {e}")
    
    if all_cached_data:
        combined_df = pd.concat(all_cached_data, ignore_index=True)
        print(f"\n📊 Всего из кэша: {len(combined_df)} записей")
        return combined_df
    
    return None

# 6. ПРОВЕРЯЕМ КЭШ И СБИРАЕМ ДАННЫЕ
print("\n" + "="*60)
print("НАЧИНАЮ СБОР ИСТОРИЧЕСКИХ ПОЧАСОВЫХ ДАННЫХ")
print("="*60)

print("\n🔍 Что вы хотите сделать?")
print("1. Собрать новые данные (2015-01-01 - 2025-12-31) и обновить кэш")
print("2. Только загрузить данные из кэша")
print("3. Удалить кэш и собрать новые данные")

choice = input("\nВыберите вариант (1/2/3, по умолчанию 1): ").strip()

all_data = []
target_end_date = datetime(2025, 12, 31)
today = datetime.now()

# Определяем фактическую конечную дату (не позже сегодняшнего дня)
actual_end_date = min(target_end_date, today)
end_date_str = actual_end_date.strftime('%Y-%m-%d')

if choice == '3':
    cache_files = [f for f in os.listdir(CACHE_DIR) if f.endswith('.pkl')]
    for cache_file in cache_files:
        os.remove(os.path.join(CACHE_DIR, cache_file))
    print(f"🗑️  Удалено {len(cache_files)} файлов из кэша")
    choice = '1'

if choice == '2':
    print(f"\n📂 Загружаю данные из кэша...")
    final_df = load_all_cached_data()
    
    if final_df is not None:
        csv_filename = f'hourly_weather_2015_{end_date_str}.csv'
        final_df.to_csv(csv_filename, index=False, encoding='utf-8-sig')
        
        print(f"\n💾 Данные сохранены в файл: '{csv_filename}'")
        print(f"📏 Размер файла: {os.path.getsize(csv_filename) / 1024:.1f} KB")
        
        print("\n📊 СТАТИСТИКА ДАННЫХ ИЗ КЭША:")
        print(f"   Записей: {len(final_df)}")
        print(f"   Городов: {final_df['city'].nunique()}")
        print(f"   Период: {final_df['time'].min()} - {final_df['time'].max()}")
        
        print("\n📋 ПЕРВЫЕ 3 СТРОКИ:")
        print(final_df.head(3))
    else:
        print("❌ Нет данных в кэше")
    
else:
    print(f"\n📅 Целевой период сбора: с 2015-01-01 по 2025-12-31")
    if actual_end_date < target_end_date:
        print(f"⚠️  Но данные доступны только до {end_date_str} (сегодня)")
    print(f"📅 Фактический период сбора: с 2015-01-01 по {end_date_str}")
    
    years_count = actual_end_date.year - 2015 + 1
    print(f"⏳ Запросов (с разбивкой): {len(cities)} городов x ~{years_count} частей = ~{len(cities) * years_count} запросов")
    print("⚠️  Важно: между запросами будут добавляться задержки для избежания блокировки")
    
    # Спрашиваем размер чанка
    chunk_size = input("\nВведите размер чанка в годах (рекомендуется 1, можно 0.5 для полугода): ").strip()
    try:
        chunk_years = float(chunk_size) if chunk_size else 1
        if chunk_years < 0.5:
            chunk_years = 0.5
            print("📌 Использую минимальный размер чанка 0.5 года")
        elif chunk_years > 2:
            print("⚠️  Рекомендуется размер не больше 2 лет, использую 2")
            chunk_years = 2
    except:
        chunk_years = 1
    
    for i, city in enumerate(cities):
        print(f"\n{'='*50}")
        print(f"Город {i+1}/{len(cities)}: {city['name']}")
        print(f"{'='*50}")
        
        # Добавляем задержку между городами
        if i > 0:
            print(f"\n⏳ Большая задержка между городами: 30 секунд...")
            time.sleep(30)
        
        df_city = get_weather_full_period_chunked(
            city['name'],
            city['lat'],
            city['lon'],
            city.get('federal_subject', 'Не указан'),
            chunk_years=chunk_years
        )
        
        if df_city is not None:
            all_data.append(df_city)
            
            if 'temperature_2m' in df_city.columns:
                temp = df_city['temperature_2m']
                print(f"\n📈 Средняя температура за весь период: {temp.mean():.1f}°C")
    
    # ОБЪЕДИНЯЕМ И СОХРАНЯЕМ ДАННЫЕ
    print("\n" + "="*60)
    print("💾 ОБРАБОТКА И СОХРАНЕНИЕ")
    print("="*60)
    
    if all_data:
        final_df = pd.concat(all_data, ignore_index=True)
        
        print(f"\n📊 Всего получено:")
        print(f"   Записей: {len(final_df)}")
        print(f"   Городов: {final_df['city'].nunique()}")
        print(f"   Период: {final_df['time'].min()} - {final_df['time'].max()}")
        
        csv_filename = f'hourly_weather_2015_{end_date_str}.csv'
        final_df.to_csv(csv_filename, index=False, encoding='utf-8-sig')
        
        print(f"\n💾 Данные сохранены в файл: '{csv_filename}'")
        print(f"📏 Размер файла: {os.path.getsize(csv_filename) / 1024:.1f} KB")
        
        print("\n📋 ПЕРВЫЕ 3 СТРОКИ:")
        print(final_df.head(3))
        
        metadata = {
            'total_cities': len(cities),
            'cities_processed': final_df['city'].nunique(),
            'total_records': len(final_df),
            'data_range': {
                'start': final_df['time'].min() if 'time' in final_df.columns else None,
                'end': final_df['time'].max() if 'time' in final_df.columns else None
            },
            'target_period': {
                'start': '2015-01-01',
                'end': '2025-12-31'
            },
            'cache_dir': CACHE_DIR,
            'cache_files': len([f for f in os.listdir(CACHE_DIR) if f.endswith('.pkl')]),
            'generated_at': datetime.now().isoformat()
        }
        
        with open(os.path.join(CACHE_DIR, 'metadata.json'), 'w', encoding='utf-8') as f:
            json.dump(metadata, f, indent=2, ensure_ascii=False)
        
    else:
        print("❌ Не удалось получить данные")

print("\n" + "="*60)
print("✅ РАБОТА ЗАВЕРШЕНА!")
print("="*60)

print(f"\n📁 СОЗДАННЫЕ ФАЙЛЫ:")
print(f"1. hourly_weather_2015_{end_date_str}.csv - Почасовые метеоданные с 2015 по {end_date_str}")
print(f"2. {CACHE_DIR}/ - Кэшированные данные")

print("\n✨ Готово!")

🌤️ Начинаю сбор исторических почасовых данных погоды...

📥 Загружаю города из selected_cities.csv...
📊 Загружено 2 городов
📋 Колонки: ['City', 'Federal subject', 'Latitude', 'Longitude']

📊 Итого для обработки: 2 городов с координатами

🌍 СПИСОК ГОРОДОВ ДЛЯ СБОРА ДАННЫХ:
📍 Kazan: 55.794649, 49.111502 (Republic of Tatarstan)
📍 Penza: 53.195348, 45.019044 (Penza Oblast)

🚀 НАЧИНАЮ СБОР ИСТОРИЧЕСКИХ ПОЧАСОВЫХ ДАННЫХ

🔍 Что вы хотите сделать?
1. Собрать новые данные (2015-01-01 - 2025-12-31) и обновить кэш
2. Только загрузить данные из кэша
3. Удалить кэш и собрать новые данные



Выберите вариант (1/2/3, по умолчанию 1):  1



📅 Целевой период сбора: с 2015-01-01 по 2025-12-31
📅 Фактический период сбора: с 2015-01-01 по 2025-12-31
⏳ Запросов (с разбивкой): 2 городов x ~11 частей = ~22 запросов
⚠️  Важно: между запросами будут добавляться задержки для избежания блокировки



Введите размер чанка в годах (рекомендуется 1, можно 0.5 для полугода):  1



Город 1/2: Kazan

📅 Полный период: 2015-01-01 - 2025-12-31
   Разбиваю на части по 1.0 году(а)

   📦 Часть 1: 2015-01-01 - 2015-12-31
   📂 Загружено из кэша (8760 записей)
   ✅ Часть 1 успешно загружена

   📦 Часть 2: 2016-01-01 - 2016-12-30
   ⏳ Задержка 2.2 сек...
   📂 Загружено из кэша (8760 записей)
   ✅ Часть 2 успешно загружена

   📦 Часть 3: 2016-12-31 - 2017-12-30
   ⏳ Задержка 2.4 сек...
   📂 Загружено из кэша (8760 записей)
   ✅ Часть 3 успешно загружена

   📦 Часть 4: 2017-12-31 - 2018-12-30
   ⏳ Задержка 2.2 сек...
   📂 Загружено из кэша (8760 записей)
   ✅ Часть 4 успешно загружена

   📦 Часть 5: 2018-12-31 - 2019-12-30
   ⏳ Задержка 4.0 сек...
   📂 Загружено из кэша (8760 записей)
   ✅ Часть 5 успешно загружена

   📦 Часть 6: 2019-12-31 - 2020-12-29
   ⏳ Задержка 3.3 сек...
   📂 Загружено из кэша (8760 записей)
   ✅ Часть 6 успешно загружена

   📦 Часть 7: 2020-12-30 - 2021-12-29
   ⏳ Задержка 3.9 сек...
   📂 Загружено из кэша (8760 записей)
   ✅ Часть 7 успешно загружен

### 5. Загрузим данные по погоде по выбранным городам в supabase

In [11]:
import pandas as pd
import requests
import json
import time
import numpy as np
from datetime import datetime

# Настройки Supabase
SUPABASE_URL = "https://xrtiwzpkjapblwcmkyve.supabase.co"
SUPABASE_KEY = "sb_publishable_9qNc84zjkn-20CUJhN_8EQ_oxdOSCzC"
TABLE_NAME = "hourly_weather_2015_2025"

# Headers
headers = {
    'apikey': SUPABASE_KEY,
    'Authorization': f'Bearer {SUPABASE_KEY}',
    'Content-Type': 'application/json',
    'Prefer': 'return=minimal'
}

print("🌤️ ЗАГРУЗЧИК ПОЛНЫХ ДАННЫХ ПОГОДЫ В SUPABASE")
print("="*60)

def check_table_exists():
    """Проверяем существование таблицы"""
    try:
        response = requests.get(
            f"{SUPABASE_URL}/rest/v1/{TABLE_NAME}",
            headers=headers,
            params={"select": "city", "limit": 1}
        )
        return response.status_code == 200
    except:
        return False

def prepare_data_for_upload(df):
    """Подготавливаем данные для загрузки в Supabase"""
    
    print("\n⚙️  Подготовка данных...")
    
    # 1. Преобразуем временные метки
    if 'time' in df.columns:
        # Конвертируем в строку в ISO формате
        df['time'] = pd.to_datetime(df['time']).dt.strftime('%Y-%m-%d %H:%M:%S+00:00')
        print(f"   ✅ Время: преобразовано {df['time'].notna().sum()} записей")
    
    if 'date' in df.columns:
        # Конвертируем дату в строку
        df['date'] = pd.to_datetime(df['date']).dt.strftime('%Y-%m-%d')
        print(f"   ✅ Дата: преобразовано {df['date'].notna().sum()} записей")
    
    # 2. Заменяем NaN на None для корректного JSON
    df = df.replace({np.nan: None, pd.NaT: None})
    
    # 3. Выводим статистику по колонкам
    print(f"\n📊 СТАТИСТИКА ДАННЫХ:")
    print(f"   Всего записей: {len(df):,}")
    print(f"   Всего колонок: {len(df.columns)}")
    
    # Группируем колонки по типам данных
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    datetime_cols = [col for col in ['time', 'date'] if col in df.columns]
    text_cols = [col for col in ['city', 'federal_subject'] if col in df.columns]
    
    print(f"   Текстовые поля: {len(text_cols)}")
    print(f"   Числовые поля: {len(numeric_cols)}")
    print(f"   Дата/время: {len(datetime_cols)}")
    
    # 4. Проверяем наличие ключевых колонок
    required_cols = ['city', 'time']
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        print(f"\n⚠️  ВНИМАНИЕ: Отсутствуют ключевые колонки: {missing_cols}")
    
    return df

def upload_data_in_batches(data, batch_size=500):
    """Загружаем данные пакетами"""
    
    total_records = len(data)
    print(f"\n🚀 Начинаю загрузку {total_records:,} записей...")
    print(f"   Размер пакета: {batch_size}")
    print(f"   Всего пакетов: {((total_records - 1) // batch_size) + 1}")
    
    uploaded = 0
    failed_batches = []
    
    for i in range(0, total_records, batch_size):
        batch_num = (i // batch_size) + 1
        total_batches = ((total_records - 1) // batch_size) + 1
        
        batch = data[i:i + batch_size]
        
        print(f"\n📤 Пакет {batch_num}/{total_batches}: {len(batch):,} записей...", end=' ')
        
        max_retries = 3
        for retry in range(max_retries):
            try:
                response = requests.post(
                    f"{SUPABASE_URL}/rest/v1/{TABLE_NAME}",
                    headers=headers,
                    json=batch,
                    timeout=30
                )
                
                if response.status_code in [200, 201, 204]:
                    uploaded += len(batch)
                    print(f"✅ Успешно ({uploaded:,}/{total_records:,})")
                    break
                elif response.status_code == 413:
                    # Слишком большой пакет, уменьшаем размер
                    print(f"⚠️  Пакет слишком большой, уменьшаю размер...")
                    batch_size = max(100, batch_size // 2)
                    continue
                elif response.status_code == 429:
                    # Слишком много запросов, ждем
                    wait_time = 2 ** retry
                    print(f"⚠️  Слишком много запросов, жду {wait_time} сек...")
                    time.sleep(wait_time)
                    continue
                else:
                    print(f"❌ Ошибка {response.status_code}")
                    print(f"   Ответ: {response.text[:200]}")
                    failed_batches.append(batch_num)
                    break
                    
            except requests.exceptions.RequestException as e:
                print(f"❌ Ошибка сети: {e}")
                if retry < max_retries - 1:
                    wait_time = 2 ** retry
                    print(f"   Повтор через {wait_time} сек...")
                    time.sleep(wait_time)
                else:
                    failed_batches.append(batch_num)
        
        # Пауза между пакетами
        if i + batch_size < total_records:
            time.sleep(0.5)
    
    return uploaded, failed_batches

def verify_upload():
    """Проверяем загруженные данные"""
    
    print("\n🔍 Проверка загруженных данных...")
    
    try:
        # Получаем количество записей
        response = requests.get(
            f"{SUPABASE_URL}/rest/v1/{TABLE_NAME}",
            headers=headers,
            params={
                "select": "city",
                "limit": 1,
                "head": "true"
            }
        )
        
        if 'content-range' in response.headers:
            count = response.headers['content-range'].split('/')[-1]
            print(f"   📊 Всего записей в таблице: {count}")
        
        # Получаем статистику по городам
        response = requests.get(
            f"{SUPABASE_URL}/rest/v1/{TABLE_NAME}",
            headers=headers,
            params={
                "select": "city,time,temperature_2m",
                "order": "time.desc",
                "limit": 3
            }
        )
        
        if response.status_code == 200:
            samples = response.json()
            print(f"\n📋 Последние записи:")
            for sample in samples:
                city = sample.get('city', 'N/A')
                time_val = sample.get('time', 'N/A')[:19] if sample.get('time') else 'N/A'
                temp = sample.get('temperature_2m', 'N/A')
                print(f"   {city} | {time_val} | {temp}°C")
        
        # Проверяем диапазон дат
        response = requests.get(
            f"{SUPABASE_URL}/rest/v1/{TABLE_NAME}",
            headers=headers,
            params={
                "select": "MIN(time),MAX(time)",
                "limit": 1
            }
        )
        
        if response.status_code == 200:
            date_range = response.json()[0]
            min_time = date_range.get('min', 'N/A')
            max_time = date_range.get('max', 'N/A')
            print(f"\n📅 Диапазон данных: {min_time} — {max_time}")
        
        return True
        
    except Exception as e:
        print(f"❌ Ошибка при проверке: {e}")
        return False

def main():
    """Основная функция"""
    
    # 1. Проверяем таблицу
    if not check_table_exists():
        print(f"❌ Таблица '{TABLE_NAME}' не существует!")
        print("\n📝 Сначала создайте таблицу в Supabase SQL Editor:")
        print("   1. Откройте SQL Editor")
        print(f"   2. Выполните SQL для создания таблицы '{TABLE_NAME}'")
        print("   3. Затем запустите этот скрипт снова")
        return
    
    print(f"✅ Таблица '{TABLE_NAME}' существует")
    
    # 2. Загружаем CSV
    csv_file = "hourly_weather_2015_2025-12-31.csv"
    print(f"\n📥 Загружаю данные из {csv_file}...")
    
    try:
        # Загружаем CSV с указанием типа данных для оптимизации
        df = pd.read_csv(csv_file, low_memory=False)
        print(f"📊 Загружено {len(df):,} записей")
        print(f"📋 Колонок: {len(df.columns)}")
        
        # Показываем первые 2 строки
        print("\n📋 Структура данных (первые 2 строки):")
        print(df.head(2).to_string())
        
        # 3. Подготавливаем данные
        df_prepared = prepare_data_for_upload(df)
        
        # 4. Конвертируем в JSON
        print("\n🔄 Конвертация в JSON...")
        data = df_prepared.to_dict('records')
        print(f"✅ Данные подготовлены для загрузки")
        
        # 5. Подтверждение
        print(f"\n⚠️  ВНИМАНИЕ: Вы собираетесь загрузить {len(data):,} записей")
        confirmation = input("Продолжить? (y/n): ").strip().lower()
        
        if confirmation != 'y':
            print("❌ Загрузка отменена")
            return
        
        # 6. Загружаем данные
        start_time = time.time()
        uploaded, failed_batches = upload_data_in_batches(data)
        elapsed_time = time.time() - start_time
        
        # 7. Отчет
        print("\n" + "="*60)
        print("📊 ОТЧЕТ О ЗАГРУЗКЕ:")
        print("="*60)
        print(f"✅ Успешно загружено: {uploaded:,} записей")
        print(f"⏱️  Время загрузки: {elapsed_time:.1f} секунд")
        print(f"📈 Скорость: {uploaded/elapsed_time:.1f} записей/сек")
        
        if failed_batches:
            print(f"❌ Проблемные пакеты: {failed_batches}")
        else:
            print("✨ Все данные успешно загружены!")
        
        # 8. Проверяем загрузку
        if uploaded > 0:
            verify_upload()
        
    except FileNotFoundError:
        print(f"❌ Файл {csv_file} не найден")
        print("📝 Поместите CSV файл с данными погоды рядом со скриптом")
    except pd.errors.EmptyDataError:
        print(f"❌ Файл {csv_file} пустой")
    except Exception as e:
        print(f"❌ Критическая ошибка: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()
    print("\n" + "="*60)
    print("🏁 РАБОТА ЗАВЕРШЕНА!")
    print("="*60)

🌤️ ЗАГРУЗЧИК ПОЛНЫХ ДАННЫХ ПОГОДЫ В SUPABASE
✅ Таблица 'hourly_weather_2015_2025' существует

📥 Загружаю данные из hourly_weather_2015_2025-12-31.csv...
📊 Загружено 192,864 записей
📋 Колонок: 39

📋 Структура данных (первые 2 строки):
    city        federal_subject   latitude  longitude              time  temperature_2m  relative_humidity_2m  dew_point_2m  apparent_temperature  pressure_msl  surface_pressure  precipitation  rain  snowfall  cloud_cover  cloud_cover_low  cloud_cover_mid  cloud_cover_high  shortwave_radiation  direct_radiation  diffuse_radiation  direct_normal_irradiance  global_tilted_irradiance  terrestrial_radiation  wind_speed_10m  wind_speed_100m  wind_direction_10m  wind_direction_100m  wind_gusts_10m  et0_fao_evapotranspiration  soil_temperature_0_to_7cm  soil_temperature_7_to_28cm  soil_temperature_28_to_100cm  soil_temperature_100_to_255cm  soil_moisture_0_to_7cm  soil_moisture_7_to_28cm  soil_moisture_28_to_100cm  soil_moisture_100_to_255cm        date
0  Kazan 

Продолжить? (y/n):  y



🚀 Начинаю загрузку 192,864 записей...
   Размер пакета: 500
   Всего пакетов: 386

✅ Успешно (500/192,864)исей... 

✅ Успешно (1,000/192,864)ей... 

✅ Успешно (1,500/192,864)ей... 

✅ Успешно (2,000/192,864)ей... 

✅ Успешно (2,500/192,864)ей... 

✅ Успешно (3,000/192,864)ей... 

✅ Успешно (3,500/192,864)ей... 

✅ Успешно (4,000/192,864)ей... 

✅ Успешно (4,500/192,864)ей... 

✅ Успешно (5,000/192,864)сей... 

✅ Успешно (5,500/192,864)сей... 

✅ Успешно (6,000/192,864)сей... 

✅ Успешно (6,500/192,864)сей... 

✅ Успешно (7,000/192,864)сей... 

✅ Успешно (7,500/192,864)сей... 

✅ Успешно (8,000/192,864)сей... 

✅ Успешно (8,500/192,864)сей... 

✅ Успешно (9,000/192,864)сей... 

✅ Успешно (9,500/192,864)сей... 

✅ Успешно (10,000/192,864)ей... 

✅ Успешно (10,500/192,864)ей... 

✅ Успешно (11,000/192,864)ей... 

✅ Успешно (11,500/192,864)ей... 

✅ Успешно (12,000/192,864)ей... 

✅ Успешно (12,500/192,864)ей... 

✅ Успешно (13,000/192,864)ей... 

✅ Успешно (13,500/192,864)ей... 

✅ Успеш

### 6. Собираем(парсим) данные по ДТП по выбранным городам с 2015 по 2025

In [19]:
import requests
import json
from datetime import datetime
import time
import os

# Конфигурация
URL = "http://stat.gibdd.ru/map/getDTPCardData"

# Города
CITIES = [
    {"name": "Пенза", "region_id": "56", "district_id": "56401"},
    {"name": "Казань", "region_id": "92", "district_id": "92401"}
]

# Заголовки
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Accept": "application/json, text/plain, */*",
    "Content-Type": "application/json",
    "Origin": "http://stat.gibdd.ru",
    "Referer": "http://stat.gibdd.ru/",
    "Accept-Language": "ru-RU,ru;q=0.9,en-US;q=0.8,en;q=0.7"
}

FIELD_NAMES = [
    "dat", "time", "coordinates", "infoDtp", "k_ul", "dor", "ndu",
    "k_ts", "ts_info", "pdop", "pog", "osv", "s_pch", "s_pog",
    "n_p", "n_pg", "obst", "sdor", "t_osv", "t_p", "t_s", "v_p", "v_v"
]

def fetch_month_data(city, year, month):
    """
    Получает данные за конкретный месяц
    """
    
    payload = {
        "data": {
            "date": [f"MONTHS:{month}.{year}"],
            "ParReg": city['district_id'][:2],
            "order": {"type": "1", "fieldName": "dat"},
            "reg": city['district_id'],
            "ind": "1",
            "st": "1",
            "en": "10000",
            "fil": {"isSummary": False},
            "fieldNames": FIELD_NAMES
        }
    }
    
    request_data = {
        "data": json.dumps(payload["data"], separators=(',', ':'))
    }
    
    try:
        response = requests.post(URL, json=request_data, headers=HEADERS, timeout=30)
        
        if response.status_code == 200:
            response_data = response.json()
            
            if "data" in response_data and response_data["data"]:
                try:
                    inner_data = json.loads(response_data["data"])
                    tab_data = inner_data.get("tab", [])
                    
                    return {
                        "success": True,
                        "tab_data": tab_data,
                        "record_count": len(tab_data)
                    }
                except:
                    return {"success": False, "error": "Ошибка парсинга"}
            else:
                return {"success": False, "error": "Нет данных"}
        else:
            return {"success": False, "error": f"HTTP {response.status_code}"}
            
    except Exception as e:
        return {"success": False, "error": str(e)}

def collect_all_data():
    """
    Сбор данных с 2015 по 2026 год
    """
    print("=" * 60)
    print("🚦 Сбор статистики ДТП с stat.gibdd.ru")
    print("=" * 60)
    print(f"📅 Период: 2015-2026")
    print(f"🏙️  Города: {', '.join([c['name'] for c in CITIES])}")
    print("=" * 60)
    
    results = {
        "metadata": {
            "source": "stat.gibdd.ru",
            "collected_at": datetime.now().isoformat(),
            "cities": [city["name"] for city in CITIES],
            "years": list(range(2015, 2027))
        },
        "data": []
    }
    
    total_requests = 0
    successful_requests = 0
    total_records = 0
    
    for city in CITIES:
        print(f"\n{'='*50}")
        print(f"🏙️  {city['name']}")
        print(f"{'='*50}")
        
        for year in range(2015, 2027):
            print(f"\n📅 {year}")
            
            for month in range(1, 13):
                # Проверка на будущие месяцы
                if year == 2026 and month > datetime.now().month:
                    print(f"   ⏹️  Месяц {month:02d} ещё не доступен")
                    break
                
                print(f"   {month:02d}: ", end="", flush=True)
                
                data = fetch_month_data(city, year, month)
                total_requests += 1
                
                if data.get("success"):
                    successful_requests += 1
                    records = data.get('record_count', 0)
                    total_records += records
                    
                    results["data"].append({
                        "city": city["name"],
                        "year": year,
                        "month": month,
                        "record_count": records,
                        "data": data.get('tab_data', [])
                    })
                    
                    print(f"✅ {records} зап.")
                else:
                    print(f"❌ {data.get('error', 'Ошибка')}")
                
                # Небольшая задержка
                time.sleep(0.3)
    
    results["metadata"]["total_requests"] = total_requests
    results["metadata"]["successful_requests"] = successful_requests
    results["metadata"]["total_records"] = total_records
    
    return results

def save_to_json(data, filename="gibdd_statistics.json"):
    """
    Сохраняет данные в JSON
    """
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2, default=str)
    
    file_size = os.path.getsize(filename) / 1024
    print(f"\n{'='*50}")
    print(f"✅ Данные сохранены в файл: {filename}")
    print(f"   Размер файла: {file_size:.2f} KB")
    print(f"   Всего записей: {data['metadata']['total_records']}")
    print(f"{'='*50}")

def main():
    try:
        results = collect_all_data()
        save_to_json(results)
        
        if results['metadata']['total_requests'] > 0:
            success_rate = (results['metadata']['successful_requests'] / 
                           results['metadata']['total_requests'] * 100)
            print(f"\n📊 Итоговая статистика:")
            print(f"   Успешных запросов: {results['metadata']['successful_requests']}")
            print(f"   Всего запросов: {results['metadata']['total_requests']}")
            print(f"   Процент успеха: {success_rate:.1f}%")
            print(f"   Всего записей о ДТП: {results['metadata']['total_records']}")
            
    except KeyboardInterrupt:
        print("\n\n⚠️  Прерывание пользователем")
        if 'results' in locals():
            save_to_json(results, "gibdd_statistics_partial.json")
    except Exception as e:
        print(f"\n❌ Критическая ошибка: {e}")

if __name__ == "__main__":
    main()

🚦 Сбор статистики ДТП с stat.gibdd.ru
📅 Период: 2015-2026
🏙️  Города: Пенза, Казань

🏙️  Пенза

📅 2015
✅ 61 зап.
   02: ✅ 68 зап.
   03: ✅ 51 зап.
   04: ✅ 56 зап.
   05: ✅ 85 зап.
   06: ✅ 95 зап.
   07: ✅ 71 зап.
   08: ✅ 93 зап.
   09: ✅ 109 зап.
   10: ✅ 97 зап.
   11: ✅ 87 зап.
   12: ✅ 94 зап.

📅 2016
✅ 73 зап.
   02: ✅ 70 зап.
   03: ✅ 55 зап.
   04: ✅ 53 зап.
   05: ✅ 69 зап.
   06: ✅ 81 зап.
   07: ✅ 82 зап.
   08: ✅ 81 зап.
   09: ✅ 105 зап.
   10: ✅ 73 зап.
   11: ✅ 88 зап.
   12: ✅ 84 зап.

📅 2017
✅ 67 зап.
   02: ✅ 71 зап.
   03: ✅ 61 зап.
   04: ✅ 45 зап.
   05: ✅ 76 зап.
   06: ✅ 74 зап.
   07: ✅ 89 зап.
   08: ✅ 96 зап.
   09: ✅ 84 зап.
   10: ✅ 96 зап.
   11: ✅ 89 зап.
   12: ✅ 84 зап.

📅 2018
✅ 73 зап.
   02: ✅ 66 зап.
   03: ✅ 65 зап.
   04: ✅ 43 зап.
   05: ✅ 65 зап.
   06: ✅ 73 зап.
   07: ✅ 69 зап.
   08: ✅ 71 зап.
   09: ✅ 80 зап.
   10: ✅ 98 зап.
   11: ✅ 85 зап.
   12: ✅ 123 зап.

📅 2019
✅ 73 зап.
   02: ✅ 54 зап.
   03: ✅ 74 зап.
   04: ✅ 54 зап.
   05: ✅ 71 з

### 7. Загрузим данные по ДТП по выбранным городам в supabase

In [23]:
import json
from supabase import create_client, Client
import os
from datetime import datetime

# Конфигурация Supabase (получите в настройках проекта)
SUPABASE_URL = "https://xrtiwzpkjapblwcmkyve.supabase.co"
SUPABASE_KEY = "sb_publishable_9qNc84zjkn-20CUJhN_8EQ_oxdOSCzC"

def init_supabase() -> Client:
    """Инициализация подключения к Supabase"""
    return create_client(SUPABASE_URL, SUPABASE_KEY)

def load_json_file(filename: str) -> dict:
    """Загружает JSON файл"""
    with open(filename, 'r', encoding='utf-8') as f:
        return json.load(f)

def upload_to_supabase(supabase: Client, data: dict):
    """Загружает данные в Supabase"""
    
    print("=" * 60)
    print("📤 Загрузка данных в Supabase")
    print("=" * 60)
    
    metadata = data.get('metadata', {})
    records = data.get('data', [])
    
    print(f"📊 Всего записей для загрузки: {len(records)}")
    print(f"📅 Период: {metadata.get('years')}")
    print(f"🏙️  Города: {metadata.get('cities')}")
    
    successful = 0
    failed = 0
    
    for idx, record in enumerate(records, 1):
        try:
            # Подготавливаем данные для вставки
            row = {
                'city': record['city'],
                'year': record['year'],
                'month': record['month'],
                'record_count': record.get('record_count', 0),
                'data': record.get('data', [])
            }
            
            # Вставляем или обновляем запись
            result = supabase.table('dtp_statistics').upsert(
                row, 
                on_conflict='city,year,month'
            ).execute()
            
            successful += 1
            print(f"✅ [{idx}/{len(records)}] {record['city']} {record['year']}-{record['month']:02d}: {record.get('record_count', 0)} записей")
            
        except Exception as e:
            failed += 1
            print(f"❌ [{idx}/{len(records)}] Ошибка: {e}")
    
    print("\n" + "=" * 60)
    print(f"📊 Итог:")
    print(f"   Успешно: {successful}")
    print(f"   Ошибок: {failed}")
    print(f"   Всего: {len(records)}")
    print("=" * 60)

def check_data(supabase: Client):
    """Проверяет загруженные данные"""
    
    print("\n🔍 Проверка данных в Supabase")
    print("-" * 40)
    
    # Общая статистика
    result = supabase.table('dtp_statistics').select('*').execute()
    total = len(result.data)
    
    print(f"📊 Всего записей в БД: {total}")
    
    # Статистика по городам
    for city in ['Пенза', 'Казань']:
        city_data = supabase.table('dtp_statistics')\
            .select('*')\
            .eq('city', city)\
            .execute()
        
        total_records = sum(r.get('record_count', 0) for r in city_data.data)
        print(f"\n🏙️  {city}:")
        print(f"   Месяцев: {len(city_data.data)}")
        print(f"   Всего ДТП: {total_records}")

def main():
    # Инициализация
    supabase = init_supabase()
    
    # Загрузка данных
    data = load_json_file('gibdd_statistics.json')
    upload_to_supabase(supabase, data)
    
    # Проверка
    check_data(supabase)

if __name__ == "__main__":
    main()

📤 Загрузка данных в Supabase
📊 Всего записей для загрузки: 266
📅 Период: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]
🏙️  Города: ['Пенза', 'Казань']
✅ [1/266] Пенза 2015-01: 61 записей
✅ [2/266] Пенза 2015-02: 68 записей
✅ [3/266] Пенза 2015-03: 51 записей
✅ [4/266] Пенза 2015-04: 56 записей
✅ [5/266] Пенза 2015-05: 85 записей
✅ [6/266] Пенза 2015-06: 95 записей
✅ [7/266] Пенза 2015-07: 71 записей
✅ [8/266] Пенза 2015-08: 93 записей
✅ [9/266] Пенза 2015-09: 109 записей
✅ [10/266] Пенза 2015-10: 97 записей
✅ [11/266] Пенза 2015-11: 87 записей
✅ [12/266] Пенза 2015-12: 94 записей
✅ [13/266] Пенза 2016-01: 73 записей
✅ [14/266] Пенза 2016-02: 70 записей
✅ [15/266] Пенза 2016-03: 55 записей
✅ [16/266] Пенза 2016-04: 53 записей
✅ [17/266] Пенза 2016-05: 69 записей
✅ [18/266] Пенза 2016-06: 81 записей
✅ [19/266] Пенза 2016-07: 82 записей
✅ [20/266] Пенза 2016-08: 81 записей
✅ [21/266] Пенза 2016-09: 105 записей
✅ [22/266] Пенза 2016-10: 73 записей
✅ [23/266] Пенз

### 8. В supabase создали основную таблицу основную таблицу по ДТП(dtp_main), таблицу с деталями по ДТП(dtp_details), таблицу с данными по транспортным средствам(dtp_vehicles), таблицу с участниками(dtp_participants)

### 9. В supabase создать пользователя для подключения к YandexDataLens

### 10. В Datalens создать новый проект, подключение к supabase. 


Ссылка на дэшборд: https://datalens.yandex/z0mgvo9lcsn2j

### 11. Загрузим данные с supabase, проанализируем, создадим дашборды в Datalens

### 12. Общий вывод


Мы собрали данные по всем городам Рф, включая название, федеральный субъект и координаты. Из этого списка выбрали два города для дальнейшего исследования. Для выбранных городов выгрузили данне по погоде с 2015 по 2025 год. Далее был создан проект в supabase куда были загружены данные по выбранным городам(название, координаты, погода). Параллельно мы собрали данные по ДТП с 2015 по 2025 год по этим городам и также загрузили в supabase.
На основе полученных данных были созданые следующие таблицы в supabase: основная информация по ДТП, детали по ДТП, таблица с транспортными средствами, таблица с участникаи ДТП. 

Полученные таблицы были загружены в Yandex DataLens для дальнейшего анализа. Мы создали несколько датасетов и чартов, и на основе их сделади дашборды.